# Themenanalyse kommunaler Bürgerhinweise

Prototyp zur Analyse von Meldetexten des Mängelmelders Konstanz mit LDA und NMF.

Fehlende und leere Texte werden getrennt behandelt. Kurze Meldungen bleiben erhalten, sofern nach dem Text Preprocessing mindestens ein Token mit zwei oder mehr Buchstaben verbleibt. Meldungen ohne Merkmale im Vokabular gehen nicht in das Modelltraining ein und werden als nicht zugeordnet ausgewiesen.

## Offene Punkte

- Die Auswahl der Stopwörter noch prüfen.
- Verschiedene Vorgaben für die Anzahl der Topics systematisch vergleichen.
- Themenbezeichnungen und Beispiele anhand der aktuellen Ergebnisse prüfen.

Die Prozentanteile beziehen sich auf alle Meldungen nach dem Text Preprocessing, einschließlich der nicht zugeordneten Meldungen. Der Anteil ohne Zuordnung wird separat ausgewiesen.

Die Quelldatei `maengelmelder_konstanz_final_9_neu.csv` wird unter `data/raw/` erwartet. Die Zellen sind von oben nach unten auszuführen. Die technische Auswertung läuft vollständig durch. Die beschriftete Übersicht und ihr Diagramm bleiben bis zur Prüfung der Themenbezeichnungen deaktiviert.


In [16]:
# Schritt 01 Umgebung und Arbeitsordner prüfen

from pathlib import Path
import sys
import re

import pandas as pd
import numpy as np
import sklearn
from IPython.display import display

print("Python executable:")
print(sys.executable)

print("\nPandas Version:")
print(pd.__version__)

print("\nNumPy Version:")
print(np.__version__)

print("\nScikit Learn Version:")
print(sklearn.__version__)

print("\nAktueller Arbeitsordner:")
print(Path.cwd())


Python executable:
C:\Users\Seb\anaconda3\envs\portfolio_da\python.exe

Pandas Version:
3.0.5

NumPy Version:
2.4.6

Scikit Learn Version:
1.9.0

Aktueller Arbeitsordner:
C:\Users\Seb\Documents\GitHub\portfolio-data-analysis\notebooks


In [17]:
# Schritt 02 Konfiguration

CSV_NAME = "maengelmelder_konstanz_final_9_neu.csv"
TEXT_COLUMN = "text_raw"
ID_COLUMN = "id"

# Kurze Meldungen bleiben erhalten, sofern nach dem Preprocessing ein Token verbleibt.
MIN_WORDS_RAW = 1
MIN_TOKENS_CLEAN = 1
MIN_TOKEN_LENGTH = 2

# Parameter der Vektorisierung
MIN_DF = 5
MAX_DF = 0.90
NGRAM_RANGE = (1, 1)
TOKEN_PATTERN = r"(?u)\b[a-zäöüß][a-zäöüß]+\b"
SUBLINEAR_TF = True

# Bisherige Modelleinstellung, die Auswahl der Topic-Anzahl steht noch aus.
N_TOPICS = 10
RANDOM_STATE = 42
LDA_MAX_ITER = 20
NMF_MAX_ITER = 500

# Erst nach der inhaltlichen Prüfung der Bezeichnungen für das aktuelle Modell aktivieren.
TOPIC_LABELS_REVIEWED = False

# Deutsche Stopwörter aus dem bisherigen Prototyp
GERMAN_STOPWORDS = {
    "aber", "alle", "allem", "allen", "aller", "alles", "als", "also", "am", "an",
    "andere", "anderem", "anderen", "anderer", "anderes", "auch", "auf", "aus",
    "bei", "bin", "bis", "bist", "da", "damit", "dann", "der", "den", "des",
    "dem", "die", "das", "dass", "daß", "dazu", "dein", "deine", "deinem",
    "deinen", "deiner", "deines", "denn", "dich", "dir", "du", "dies", "diese",
    "diesem", "diesen", "dieser", "dieses", "doch", "dort", "durch", "ein",
    "eine", "einem", "einen", "einer", "eines", "einige", "einigen", "einiger",
    "einiges", "einmal", "er", "ihn", "ihm", "es", "etwas", "euer", "eure",
    "eurem", "euren", "eurer", "eures", "für", "gegen", "gewesen", "hab",
    "habe", "haben", "hat", "hatte", "hatten", "hier", "hin", "hinter", "ich",
    "mich", "mir", "ihr", "ihre", "ihrem", "ihren", "ihrer", "ihres", "euch",
    "im", "in", "indem", "ins", "ist", "jede", "jedem", "jeden", "jeder",
    "jedes", "jene", "jenem", "jenen", "jener", "jenes", "jetzt", "kann",
    "kein", "keine", "keinem", "keinen", "keiner", "keines", "können", "könnte",
    "machen", "man", "manche", "manchem", "manchen", "mancher", "manches",
    "mein", "meine", "meinem", "meinen", "meiner", "meines", "mit", "muss",
    "musste", "nach", "nicht", "nichts", "noch", "nun", "nur", "ob", "oder",
    "ohne", "sehr", "sein", "seine", "seinem", "seinen", "seiner", "seines",
    "selbst", "sich", "sie", "ihnen", "sind", "so", "solche", "solchem",
    "solchen", "solcher", "solches", "soll", "sollte", "sondern", "sonst",
    "über", "um", "und", "uns", "unser", "unsere", "unserem", "unseren",
    "unserer", "unseres", "unter", "viel", "vom", "von", "vor", "während",
    "war", "waren", "warst", "was", "weg", "weil", "weiter", "welche",
    "welchem", "welchen", "welcher", "welches", "wenn", "werde", "werden",
    "wie", "wieder", "will", "wir", "wird", "wirst", "wo", "wollen", "wollte",
    "würde", "würden", "zu", "zum", "zur", "zwar", "zwischen"
}

# Zusammengeführte Ergänzungen aus den bisherigen Versuchen.
DOMAIN_STOPWORDS = {
    "abends", "abgeholt", "abgelaufen", "abgelegt", "abgeschlossen", "abgestellt",
    "abgestellte", "abgestellter", "abgestelltes", "abholung", "adresssuche", "aktuell",
    "allerdings", "allmannsdorf", "allmannshoehe", "alte", "alter", "altes", "andere",
    "anderen", "ausgefallen", "autom", "beethovenstr", "befinden", "befindet", "beiden", "beim",
    "bereich", "bereits", "beschaedigt", "beschädigt", "bild", "bilder", "bitte", "brennen",
    "brennt", "ca", "circa", "dank", "danke", "dankeschoen", "dankeschön", "defekt", "defekte",
    "defekten", "defekter", "dienstag", "direkt", "donnerstag", "dorffriedhofsweg", "dunkel",
    "ecke", "entfernen", "entsorgt", "ermittelt", "etc", "flurweg", "foto", "fotos", "freitag",
    "frieda", "fuer", "funktion", "funktioniert", "fuss", "ganze", "gegenueber", "gegenüber",
    "gehen", "geht", "gemeldet", "gestern", "guten", "haengt", "hallo", "hans",
    "hansegartenstrasse", "haus", "hausnummer", "heugaessle", "heute", "hinten", "hoehe",
    "hoehenweg", "hängt", "höhe", "immer", "jahr", "jahren", "kaputt", "kaputte", "kaputtes",
    "kaum", "koennen", "koennte", "kommen", "kommt", "komplett", "konstanz", "konstanzer",
    "kreuzlinger", "können", "könnte", "laengerem", "leider", "leuchte", "leuchten", "leuchtet",
    "liegen", "liegt", "litzelstetten", "macairestrasse", "mainaustrasse", "mal", "mannheimer",
    "max", "mehr", "mehrere", "mehreren", "mehrfach", "mfg", "mindestens", "mittags",
    "mittwoch", "moeglich", "moegliche", "moeglicherweise", "monat", "monaten", "montag",
    "morgen", "möglich", "nacht", "naehe", "nahe", "neben", "nr", "nummer", "oben", "ort",
    "peter", "richtung", "samstag", "schaden", "schadhaft", "schaeden", "schneckenburgstrasse",
    "schoen", "schon", "schwach", "schön", "seit", "seite", "siehe", "sieht", "sigrist",
    "sonnentauweg", "sonntag", "stadt", "stark", "stehen", "steht", "steinstrasse", "stelle",
    "str", "stromeyer", "tag", "tage", "tagen", "teil", "teile", "thumb", "ueber", "unten",
    "untere", "unteren", "urisberg", "vermutlich", "viele", "vielen", "vorne", "waere",
    "waeren", "weiteres", "woche", "wochen", "worden", "wuerde", "wuerden", "wurde", "wurden",
    "wäre", "wären", "zeit", "zwei",
}

STOPWORDS_FINAL = GERMAN_STOPWORDS | DOMAIN_STOPWORDS

# Bisherige Zuordnung ausgewählter Schreibweisen und Begriffe
CANONICAL_TERMS_FINAL = {
    "abfaelle": "müll",
    "abfall": "müll",
    "abfälle": "müll",
    "aeste": "äste",
    "baeume": "baum",
    "beschaedigte": "beschädigt",
    "beschaedigten": "beschädigt",
    "beschädigte": "beschädigt",
    "bäume": "baum",
    "container": "müllcontainer",
    "fahrrads": "fahrrad",
    "fahrraeder": "fahrrad",
    "fahrräder": "fahrrad",
    "fuer": "für",
    "fussgaenger": "fußgänger",
    "fussgaengerampel": "ampel",
    "fussweg": "gehweg",
    "fusswege": "gehweg",
    "fußgängerampel": "ampel",
    "fußweg": "gehweg",
    "fußwege": "gehweg",
    "gefaehrlich": "gefährlich",
    "gehwege": "gehweg",
    "grobmuell": "grobmüll",
    "gruen": "grün",
    "gullideckel": "gulli",
    "gullis": "gulli",
    "haengt": "hängt",
    "hecken": "hecke",
    "heckenwuchs": "hecke",
    "koennen": "können",
    "laeuft": "läuft",
    "lampe": "beleuchtung",
    "lampen": "beleuchtung",
    "laterne": "beleuchtung",
    "laternen": "beleuchtung",
    "moeglich": "möglich",
    "muell": "müll",
    "muelleimer": "mülleimer",
    "müllcontainer": "müll",
    "mülltonne": "müll",
    "mülltonnen": "müll",
    "radwege": "radweg",
    "schilder": "schild",
    "straeucher": "sträucher",
    "strasse": "straße",
    "strassen": "straße",
    "strassenbeleuchtung": "beleuchtung",
    "strassenlampe": "beleuchtung",
    "strassenlampen": "beleuchtung",
    "strassenlaterne": "beleuchtung",
    "strassenlaternen": "beleuchtung",
    "strassenleuchte": "beleuchtung",
    "strassenleuchten": "beleuchtung",
    "strassenseiten": "straßenseiten",
    "strassensteiten": "straßenseiten",
    "strauch": "sträucher",
    "straßenbeleuchtung": "beleuchtung",
    "straßenlaterne": "beleuchtung",
    "straßenlaternen": "beleuchtung",
    "ueber": "über",
    "ueberfuellt": "überfüllt",
    "umgestuerzt": "umgestürzt",
    "verkehrsschild": "schild",
    "verkehrsschilder": "schild",
    "verschmutzte": "verschmutzt",
    "verschmutzten": "verschmutzt",
    "verunreinigt": "verschmutzt",
    "waechst": "wächst",
    "waere": "wäre",
    "zerstoert": "zerstört",
}


In [18]:
# Schritt 03 CSV-Datei laden

from pathlib import Path
import pandas as pd

possible_paths = [
    Path("data") / "raw" / CSV_NAME,
    Path("..") / "data" / "raw" / CSV_NAME,
]

csv_path = None

for path in possible_paths:
    print("Prüfe:", path.resolve())
    if path.exists():
        csv_path = path
        break

if csv_path is None:
    raise FileNotFoundError(
        "CSV wurde nicht gefunden. Prüfe, ob die Datei im Ordner data/raw liegt."
    )

print("\nGefundene CSV:")
print(csv_path.resolve())

# Der Projektordner ergibt sich aus dem Pfad data/raw der gefundenen CSV.
PROJECT_DIR = csv_path.parent.resolve().parent.parent

df = pd.read_csv(csv_path)

print("\nShape:")
print(df.shape)

print("\nSpalten:")
print(list(df.columns))

df.head(3)


Prüfe: C:\Users\Seb\Documents\GitHub\portfolio-data-analysis\notebooks\data\raw\maengelmelder_konstanz_final_9_neu.csv
Prüfe: C:\Users\Seb\Documents\GitHub\portfolio-data-analysis\data\raw\maengelmelder_konstanz_final_9_neu.csv

Gefundene CSV:
C:\Users\Seb\Documents\GitHub\portfolio-data-analysis\data\raw\maengelmelder_konstanz_final_9_neu.csv

Shape:
(2795, 10)

Spalten:
['id', 'date_created', 'text_raw', 'category_parent', 'category_child', 'responsible', 'lat', 'lon', 'address_input', 'detail_url']


,id,date_created,text_raw,category_parent,category_child,responsible,lat,lon,address_input,detail_url
0,2794231,2026-06-02,Seit der Nacht von Sonntag auf Montag steht im...,Muell/Sauberkeit,Illegaler Muell,Technische Betriebe,47.6661,9.1733,keine (autom. ermittelt: Frieda-Sigrist-Weg 13...,https://konstanz-mitgestalten.de/bms/2794231
1,2794197,2026-06-02,In der Skaterbahn des urisberg Spielplatz befi...,Oeffentliche Orte/Ufer,Freizeitgelaende beschaedigt / verunreinigt,Technische Betriebe,47.6907,9.1518,keine (autom. ermittelt: Benedikt-Bauer-Strass...,https://konstanz-mitgestalten.de/bms/2794197
2,2793929,2026-06-02,permanentes rausstellen von Grobmuell/ Altklei...,Muell/Sauberkeit,verschmutzte Strassen und Plaetze,Technische Betriebe,47.6844,9.1546,"keine (autom. ermittelt: Freibuergleweg 6, 784...",https://konstanz-mitgestalten.de/bms/2793929


In [19]:
# Schritt 04 Data Profiling

required_columns = [
    "id",
    "date_created",
    TEXT_COLUMN,
    "category_parent",
    "category_child",
    "responsible",
    "lat",
    "lon",
    "address_input",
    "detail_url",
]

missing_required = [col for col in required_columns if col not in df.columns]

if missing_required:
    raise ValueError(f"Folgende erwartete Spalten fehlen: {missing_required}")

print("Anzahl Zeilen:", len(df))

print("\nFehlende Werte je Spalte:")
print(df.isna().sum())

print("\nFehlende Werte in text_raw:")
print(df[TEXT_COLUMN].isna().sum())

print("\nDubletten in text_raw:")
print(df[TEXT_COLUMN].duplicated().sum())

print("\nKategorien in category_parent:")
print(df["category_parent"].value_counts(dropna=False))

print("\nKategorien in category_child:")
print(df["category_child"].value_counts(dropna=False).head(25))

print("\nTextlängen in Wörtern:")
word_counts_raw = df[TEXT_COLUMN].dropna().astype(str).str.split().str.len()
print(word_counts_raw.describe())

print("\nBeispiele aus text_raw:")
for i, text in enumerate(df[TEXT_COLUMN].dropna().astype(str).head(5), start=1):
    print(f"\nBeispiel {i}:")
    print(text[:500])


Anzahl Zeilen: 2795

Fehlende Werte je Spalte:
id                 0
date_created       0
text_raw           0
category_parent    0
category_child     0
responsible        0
lat                2
lon                2
address_input      0
detail_url         0
dtype: int64

Fehlende Werte in text_raw:
0

Dubletten in text_raw:
113

Kategorien in category_parent:
category_parent
Strassen, Fahrrad & Verkehr    982
Beleuchtung                    727
Muell/Sauberkeit               682
Pflanzenwuchs                  185
Oeffentliche Orte/Ufer         174
Hinweise                        45
Name: count, dtype: int64

Kategorien in category_child:
category_child
Strassenlaternen defekt                         727
Illegaler Muell                                 424
Fahrrad gefunden                                260
verschmutzte Strassen und Plaetze               217
Verkehrsbehindernder Heckenwuchs, Straeucher    185
Schilder unkenntlich                            150
Abgemeldete Fahrzeuge        

In [20]:
# Schritt 05 Arbeitsdatensatz erstellen

def prepare_records(
    source_df: pd.DataFrame,
    text_column: str,
    min_words: int,
    id_column: str,
):
    """Doppelte Datensätze entfernen und Meldungen mit unterschiedlichen IDs beibehalten."""
    if not source_df.columns.is_unique:
        raise ValueError("Die Spaltennamen müssen eindeutig sein.")
    missing_columns = [col for col in (text_column, id_column) if col not in source_df.columns]
    if missing_columns:
        raise ValueError(f"Erforderliche Spalten fehlen: {missing_columns}")
    if text_column == id_column:
        raise ValueError("Textspalte und ID-Spalte müssen unterschiedlich sein.")
    if not isinstance(min_words, int) or isinstance(min_words, bool) or min_words < 1:
        raise ValueError("min_words muss eine ganze Zahl größer oder gleich 1 sein.")

    counts = {"input": len(source_df)}

    # Fehlende IDs nicht als gemeinsame Meldung behandeln.
    id_keys = source_df[id_column].astype("string").str.strip()
    if id_keys.isna().any() or id_keys.eq("").any():
        raise ValueError("Meldungs-IDs fehlen. Die betroffenen Datensätze müssen geprüft werden.")

    # Nur über alle Spalten identische Datensätze entfernen, einschließlich der ID.
    work = source_df.drop_duplicates(keep="first").copy()
    counts["duplicates"] = len(source_df) - len(work)

    # Bei unterschiedlichen Inhalten zur selben ID keinen Datensatz willkürlich auswählen.
    if work[id_column].astype("string").str.strip().duplicated(keep=False).any():
        raise ValueError(
            "Zu einer Meldungs-ID liegen unterschiedliche Datensätze vor. "
            "Die Unterschiede müssen vor der Analyse geprüft werden."
        )

    before = len(work)
    work = work.dropna(subset=[text_column]).copy()
    counts["missing"] = before - len(work)
    work[text_column] = work[text_column].astype(str).str.strip()

    before = len(work)
    work = work.loc[work[text_column].str.len() > 0].copy()
    counts["empty"] = before - len(work)

    # Gleiche Texte mit unterschiedlichen IDs zählen weiterhin als einzelne Meldungen.
    work["word_count_raw"] = work[text_column].str.split().str.len()
    before = len(work)
    work = work.loc[work["word_count_raw"] >= min_words].copy()
    counts["raw_short"] = before - len(work)
    counts["remaining"] = len(work)
    counts["text_repetitions"] = int(work[text_column].duplicated().sum())

    if counts["input"] != sum(counts[key] for key in (
        "duplicates", "missing", "empty", "raw_short", "remaining"
    )):
        raise RuntimeError("Die dokumentierten Zeilenzahlen sind nicht konsistent.")

    return work.reset_index(drop=True), counts


df_work, record_counts = prepare_records(df, TEXT_COLUMN, MIN_WORDS_RAW, ID_COLUMN)

print("Ausgangszeilen:", record_counts["input"])
print("Entfernte vollständig identische Datensätze:", record_counts["duplicates"])
print("Entfernte fehlende Texte:", record_counts["missing"])
print("Entfernte leere Texte:", record_counts["empty"])
print("Entfernte Texte unter der Mindestlänge:", record_counts["raw_short"])
print("Meldungen im Arbeitsdatensatz:", record_counts["remaining"])
print("Weitere Meldungen mit gleichem Text, beibehalten:", record_counts["text_repetitions"])

# Unterschiedliche IDs mit gleichem Text bleiben erhalten, eine identische Zeile entfällt.
_test_data = pd.DataFrame({
    ID_COLUMN: [1, 2, 3, 4, 5, 6, 7, 8, 8],
    TEXT_COLUMN: [None, np.nan, pd.NA, "", "  ", " Müll ", "Müll", "Lampe defekt", "Lampe defekt"],
})
_test_work, _test_counts = prepare_records(_test_data, TEXT_COLUMN, 1, ID_COLUMN)
assert _test_work[ID_COLUMN].tolist() == [6, 7, 8]
assert _test_work[TEXT_COLUMN].tolist() == ["Müll", "Müll", "Lampe defekt"]
assert _test_counts == {
    "input": 9, "duplicates": 1, "missing": 3, "empty": 2,
    "raw_short": 0, "remaining": 3, "text_repetitions": 1,
}
print("Funktionstest für Fehlwerte und Dubletten: bestanden")

df_work[[ID_COLUMN, TEXT_COLUMN, "category_parent", "category_child", "word_count_raw"]].head(10)

Ausgangszeilen: 2795
Entfernte vollständig identische Datensätze: 0
Entfernte fehlende Texte: 0
Entfernte leere Texte: 0
Entfernte Texte unter der Mindestlänge: 0
Meldungen im Arbeitsdatensatz: 2795
Weitere Meldungen mit gleichem Text, beibehalten: 113
Funktionstest für Fehlwerte und Dubletten: bestanden


,id,text_raw,category_parent,category_child,word_count_raw
0,2794231,Seit der Nacht von Sonntag auf Montag steht im...,Muell/Sauberkeit,Illegaler Muell,38
1,2794197,In der Skaterbahn des urisberg Spielplatz befi...,Oeffentliche Orte/Ufer,Freizeitgelaende beschaedigt / verunreinigt,9
2,2793929,permanentes rausstellen von Grobmuell/ Altklei...,Muell/Sauberkeit,verschmutzte Strassen und Plaetze,26
3,2792591,Glas und Muell,Muell/Sauberkeit,verschmutzte Strassen und Plaetze,3
4,2792403,Schild umgestuerzt.,"Strassen, Fahrrad & Verkehr",Schilder unkenntlich,2
5,2791799,Beschaedigte Ampel auf beiden Strassensteiten ...,"Strassen, Fahrrad & Verkehr",Ampel defekt,26
6,2790891,weiteres Sofateil wurde hier abgestellt,Muell/Sauberkeit,Illegaler Muell,5
7,2790889,Teile eines Sofas wurden hier abgelegt,Muell/Sauberkeit,Illegaler Muell,6
8,2788177,Eine Parkbank wurde in der Nacht von Freitag a...,Oeffentliche Orte/Ufer,Sportplaetze beschaedigt / verunreinigt,11
9,2787373,Strassenlaterne zwischen Nr. 17A und 19 seit T...,Beleuchtung,Strassenlaternen defekt,9


In [21]:
# Schritt 06 Ausgabeordner festlegen

OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output-Ordner:")
print(OUTPUT_DIR)


Output-Ordner:
C:\Users\Seb\Documents\GitHub\portfolio-data-analysis\outputs


In [22]:
# Schritt 07 Text Preprocessing definieren

# Die Endungsregeln erkennen mögliche Ortsangaben, aber nicht alle Ortsnamen.
def is_location_token(token: str) -> bool:
    # Mögliche Straßennamen anhand der Endung erkennen und "straße" ausnehmen.
    if token == "straße":
        return False

    if token.endswith(("strasse", "straße", "str")) and len(token) > 5:
        return True

    # Mögliche Wegnamen erkennen und die folgenden allgemeinen Wegbegriffe ausnehmen.
    if token in {"gehweg", "radweg", "fahrradweg"}:
        return False

    if token.endswith("weg") and len(token) > 5:
        return True

    return False

def clean_text(text: str) -> str:
    text = str(text).lower()

    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-zäöüß\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = []

    for token in text.split():
        token = CANONICAL_TERMS_FINAL.get(token, token)

        if is_location_token(token):
            continue

        if token not in STOPWORDS_FINAL and len(token) >= MIN_TOKEN_LENGTH:
            tokens.append(token)

    return " ".join(tokens)


In [23]:
# Schritt 08 Text Preprocessing anwenden

# Für jeden Lauf eine neue Kopie des Arbeitsdatensatzes erzeugen.
df_model = df_work.copy()
df_model["text_clean"] = df_model[TEXT_COLUMN].apply(clean_text)
df_model["word_count_clean"] = df_model["text_clean"].str.split().str.len()

before_clean_filter = len(df_model)
df_model = df_model[df_model["word_count_clean"] >= MIN_TOKENS_CLEAN].copy()
after_clean_filter = len(df_model)
df_model = df_model.reset_index(drop=True)

print("Zeilen vor Textfilter:", before_clean_filter)
print("Zeilen nach Textfilter:", after_clean_filter)
print("Entfernte Meldungen:", before_clean_filter - after_clean_filter)
print("\nVerteilung der Wortanzahl nach Text Preprocessing:")
print(df_model["word_count_clean"].describe())


Zeilen vor Textfilter: 2795
Zeilen nach Textfilter: 2757
Entfernte Meldungen: 38

Verteilung der Wortanzahl nach Text Preprocessing:
count    2757.000000
mean        7.402249
std         8.178322
min         1.000000
25%         2.000000
50%         5.000000
75%        10.000000
max        97.000000
Name: word_count_clean, dtype: float64


In [24]:
# Schritt 09 Bereinigte Texte anhand von Beispielen ansehen

print("Beispiele vorher/nachher:")

for _, row in df_model[[TEXT_COLUMN, "text_clean"]].head(10).iterrows():
    print("\nRAW:")
    print(row[TEXT_COLUMN][:300])
    print("CLEAN:")
    print(row["text_clean"])


Beispiele vorher/nachher:

RAW:
Seit der Nacht von Sonntag auf Montag steht im Frieda-Sigrist-Weg 15 immer noch ein grosses Holzregal mit dem Hinweis "zum Verschenken", obwohl es schon mehrfach geregnet hat. Das Regal wird sich durch das Wasser verziehen und unbrauchbar werden.
CLEAN:
grosses holzregal hinweis verschenken obwohl geregnet regal wasser verziehen unbrauchbar

RAW:
In der Skaterbahn des urisberg Spielplatz befinden sich Scherben
CLEAN:
skaterbahn spielplatz scherben

RAW:
permanentes rausstellen von Grobmuell/ Altkleidern mehrere Kisten immer wieder...dadurch einengen des Fusweges, kein Rollstuhl, Kinderwagen etc. kommt ungehindert vorbei...die Strasse wird durch diese Muellkisten zum Muellsteig...
CLEAN:
permanentes rausstellen grobmüll altkleidern kisten dadurch einengen fusweges rollstuhl kinderwagen ungehindert vorbei straße muellkisten muellsteig

RAW:
Glas und Muell
CLEAN:
glas müll

RAW:
Schild umgestuerzt.
CLEAN:
schild umgestürzt

RAW:
Beschaedigte Ampel auf beide

In [25]:
# Schritt 10 Texte mit CountVectorizer und TfidfVectorizer vektorisieren

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

texts_clean = df_model["text_clean"].tolist()

count_vectorizer = CountVectorizer(
    min_df=MIN_DF,
    max_df=MAX_DF,
    token_pattern=TOKEN_PATTERN,
    ngram_range=NGRAM_RANGE
)

tfidf_vectorizer = TfidfVectorizer(
    min_df=MIN_DF,
    max_df=MAX_DF,
    token_pattern=TOKEN_PATTERN,
    ngram_range=NGRAM_RANGE,
    sublinear_tf=SUBLINEAR_TF
)

X_count = count_vectorizer.fit_transform(texts_clean)
X_tfidf = tfidf_vectorizer.fit_transform(texts_clean)

print("Anzahl Dokumente:", len(texts_clean))
print("Count-Matrix:", X_count.shape)
print("TF-IDF-Matrix:", X_tfidf.shape)

print("\nBeispiele aus Count-Vokabular:")
print(count_vectorizer.get_feature_names_out()[:30])

print("\nBeispiele aus TF-IDF-Vokabular:")
print(tfidf_vectorizer.get_feature_names_out()[:30])


Anzahl Dokumente: 2757
Count-Matrix: (2757, 831)
TF-IDF-Matrix: (2757, 831)

Beispiele aus Count-Vokabular:
['ab' 'abbiegen' 'abdeckung' 'abgebaut' 'abgebrochen' 'abgefallen'
 'abgeladen' 'abgemeldet' 'abgemeldeter' 'abgemeldetes' 'abhilfe'
 'ablaeufe' 'ablauf' 'absperrung' 'aeltere' 'alm' 'alten' 'altglas'
 'altglascontainer' 'altglascontainern' 'altkleider' 'altkleidercontainer'
 'ampel' 'ampelschaltung' 'anbringen' 'aneinander' 'anfang' 'angebracht'
 'angefahren' 'angeschlossen']

Beispiele aus TF-IDF-Vokabular:
['ab' 'abbiegen' 'abdeckung' 'abgebaut' 'abgebrochen' 'abgefallen'
 'abgeladen' 'abgemeldet' 'abgemeldeter' 'abgemeldetes' 'abhilfe'
 'ablaeufe' 'ablauf' 'absperrung' 'aeltere' 'alm' 'alten' 'altglas'
 'altglascontainer' 'altglascontainern' 'altkleider' 'altkleidercontainer'
 'ampel' 'ampelschaltung' 'anbringen' 'aneinander' 'anfang' 'angebracht'
 'angefahren' 'angeschlossen']


In [26]:
# Kontrolle vor der Modellierung

print("Eingelesene Meldungen:", len(df))
print("Meldungen nach Text Preprocessing:", len(df_model))
print("Count-Matrix:", X_count.shape)
print("TF-IDF-Matrix:", X_tfidf.shape)
print(
    "Meldungen ohne Merkmale in der Count-Matrix:",
    int((X_count.getnnz(axis=1) == 0).sum())
)
print(
    "Meldungen ohne Merkmale in der TF-IDF-Matrix:",
    int((X_tfidf.getnnz(axis=1) == 0).sum())
)

# Kontrollieren, dass nach dem Preprocessing keine leeren Texte verbleiben.
assert df_model[TEXT_COLUMN].notna().all()
assert (df_model["word_count_clean"] >= MIN_TOKENS_CLEAN).all()
assert len(df_model) == X_count.shape[0] == X_tfidf.shape[0]

# Kurze, inhaltliche Testmeldungen bleiben mit der neuen Mindestlänge erhalten.
assert len(clean_text("Lampe defekt").split()) >= MIN_TOKENS_CLEAN
assert len(clean_text("Müll").split()) >= MIN_TOKENS_CLEAN
assert clean_text("Bitte danke") == ""
print("Kontrolle der Textfilter: bestanden")



Eingelesene Meldungen: 2795
Meldungen nach Text Preprocessing: 2757
Count-Matrix: (2757, 831)
TF-IDF-Matrix: (2757, 831)
Meldungen ohne Merkmale in der Count-Matrix: 108
Meldungen ohne Merkmale in der TF-IDF-Matrix: 108
Kontrolle der Textfilter: bestanden


In [27]:
# Schritt 10b Meldungen ohne Merkmale erfassen

def select_model_documents(documents, count_matrix, tfidf_matrix):
    """Die gleiche Auswahl von Meldungen auf beide Merkmalsmatrizen anwenden."""
    if len(documents) != count_matrix.shape[0] or count_matrix.shape != tfidf_matrix.shape:
        raise ValueError("Meldungen und Merkmalsmatrizen passen nicht zusammen.")
    if not documents.index.is_unique:
        raise ValueError("Der Index der Meldungen muss eindeutig sein.")

    count_matrix = count_matrix.tocsr(copy=True)
    tfidf_matrix = tfidf_matrix.tocsr(copy=True)
    for matrix in (count_matrix, tfidf_matrix):
        if not np.isfinite(matrix.data).all() or (matrix.data < 0).any():
            raise ValueError("Merkmalsmatrizen müssen endliche, nichtnegative Werte enthalten.")
        matrix.eliminate_zeros()

    count_mask = np.asarray(count_matrix.getnnz(axis=1)).ravel() > 0
    tfidf_mask = np.asarray(tfidf_matrix.getnnz(axis=1)).ravel() > 0
    if not np.array_equal(count_mask, tfidf_mask):
        raise ValueError("Count und TF-IDF ergeben unterschiedliche Meldungen ohne Merkmale.")

    # Die ursprünglichen Zeilenpositionen bleiben über den Index nachvollziehbar.
    fit_documents = documents.iloc[np.flatnonzero(count_mask)].copy()
    without_features = documents.iloc[np.flatnonzero(~count_mask)].copy()
    return (
        count_mask,
        fit_documents,
        count_matrix[count_mask],
        tfidf_matrix[count_mask],
        without_features,
    )


if not np.array_equal(
    count_vectorizer.get_feature_names_out(), tfidf_vectorizer.get_feature_names_out()
):
    raise ValueError("Count und TF-IDF verwenden unterschiedliche Vokabulare.")

model_mask, df_fit, X_count_fit, X_tfidf_fit, df_without_features = select_model_documents(
    df_model, X_count, X_tfidf
)
N_ANALYSIS_DOCS = len(df_model)
if N_ANALYSIS_DOCS == 0:
    raise ValueError("Nach dem Text Preprocessing sind keine Meldungen vorhanden.")

# Vokabular und IDF bleiben aus dem gesamten Korpus nach Text Preprocessing erhalten.
# Nur für das Training werden die Zeilen ohne Merkmale ausgeschlossen.
assert len(df_fit) + len(df_without_features) == N_ANALYSIS_DOCS
assert len(df_fit) == X_count_fit.shape[0] == X_tfidf_fit.shape[0]
assert (X_count_fit.getnnz(axis=1) > 0).all()
assert (X_tfidf_fit.getnnz(axis=1) > 0).all()

print("Meldungen nach Text Preprocessing:", N_ANALYSIS_DOCS)
print("Meldungen mit Merkmalen für das Modelltraining:", len(df_fit))
print("Nicht zugeordnet, da keine Merkmale im Vokabular:", len(df_without_features))
print("Count-Matrix für das Modelltraining:", X_count_fit.shape)
print("TF-IDF-Matrix für das Modelltraining:", X_tfidf_fit.shape)
print("Meldungen ohne Merkmale in den Trainingsmatrizen:", 0)
print("Bezugsgröße der Prozentanteile:", N_ANALYSIS_DOCS)
print("Kontrolle der Auswahl: bestanden")


Meldungen nach Text Preprocessing: 2757
Meldungen mit Merkmalen für das Modelltraining: 2649
Nicht zugeordnet, da keine Merkmale im Vokabular: 108
Count-Matrix für das Modelltraining: (2649, 831)
TF-IDF-Matrix für das Modelltraining: (2649, 831)
Meldungen ohne Merkmale in den Trainingsmatrizen: 0
Bezugsgröße der Prozentanteile: 2757
Kontrolle der Auswahl: bestanden


In [28]:
# Schritt 11 LDA und NMF trainieren

from sklearn.decomposition import LatentDirichletAllocation, NMF

if len(df_fit) == 0:
    raise ValueError("Keine Meldungen mit Merkmalen vorhanden. Daten und Vokabularfilter prüfen.")
if not isinstance(N_TOPICS, int) or isinstance(N_TOPICS, bool) or N_TOPICS < 1:
    raise ValueError("N_TOPICS muss eine positive ganze Zahl sein.")
if N_TOPICS > min(X_tfidf_fit.shape):
    raise ValueError("Für diese NMF-Initialisierung gibt es zu wenige Meldungen oder Merkmale.")

lda_model = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=RANDOM_STATE,
    learning_method="batch",
    max_iter=LDA_MAX_ITER
)

nmf_model = NMF(
    n_components=N_TOPICS,
    random_state=RANDOM_STATE,
    init="nndsvda",
    max_iter=NMF_MAX_ITER
)

lda_doc_topic = lda_model.fit_transform(X_count_fit)
nmf_doc_topic = nmf_model.fit_transform(X_tfidf_fit)

print("LDA Dokument-Topic-Matrix:", lda_doc_topic.shape)
print("NMF Dokument-Topic-Matrix:", nmf_doc_topic.shape)


LDA Dokument-Topic-Matrix: (2649, 10)
NMF Dokument-Topic-Matrix: (2649, 10)


In [29]:
# Schritt 11b Topic-Zuordnung und nicht zugeordnete Meldungen erfassen

def assign_topics(doc_topic_matrix, model_mask, source_index):
    """Modellgewichte den ursprünglichen Zeilen zuordnen und fehlende Merkmale kennzeichnen."""
    weights = np.asarray(doc_topic_matrix, dtype=float)
    mask = np.asarray(model_mask)
    if mask.ndim != 1 or mask.dtype != bool or len(mask) != len(source_index):
        raise ValueError("Die Zeilenauswahl passt nicht zum Index der Meldungen.")
    if not source_index.is_unique:
        raise ValueError("Der Index der Meldungen muss eindeutig sein.")
    if weights.ndim != 2 or weights.shape[0] != int(mask.sum()) or weights.shape[1] == 0:
        raise ValueError("Die Dokument-Topic-Matrix passt nicht zur Zeilenauswahl.")
    if not np.isfinite(weights).all() or (weights < 0).any():
        raise ValueError("Topic-Gewichte müssen endlich und nichtnegativ sein.")

    result = pd.DataFrame(index=source_index.copy())
    result["Topic"] = pd.Series(pd.NA, index=source_index, dtype="Int64")
    result["Status"] = pd.Series(
        "Keine Merkmale im Vokabular", index=source_index, dtype="string"
    )
    result["Gleicher Höchstwert"] = False
    positions = np.flatnonzero(mask)
    if len(positions) == 0:
        return result

    max_weights = weights.max(axis=1)
    positive = max_weights > 0
    valid_index = source_index.take(positions[positive])
    zero_index = source_index.take(positions[~positive])

    # Nur positive Gewichte für eine Zuordnung verwenden.
    # Bei exakt gleichen Höchstwerten wählt argmax die erste Topic-Nummer.
    labels = weights[positive].argmax(axis=1) + 1
    result.loc[valid_index, "Topic"] = labels
    result.loc[valid_index, "Status"] = "Topic zugeordnet"
    result.loc[zero_index, "Status"] = "Keine positiven Topic-Gewichte"
    tied = (weights[positive] == max_weights[positive, None]).sum(axis=1) > 1
    result.loc[valid_index, "Gleicher Höchstwert"] = tied
    return result


lda_assignments = assign_topics(lda_doc_topic, model_mask, df_model.index)
nmf_assignments = assign_topics(nmf_doc_topic, model_mask, df_model.index)

# Künstlicher Test: eine zugeordnete Meldung, eine ohne Merkmale und eine ohne positive Gewichte.
_test_assignments = assign_topics(
    np.array([[0.2, 0.8], [0.0, 0.0]]),
    np.array([True, False, True]),
    pd.Index([10, 20, 30]),
)
assert _test_assignments.loc[10, "Topic"] == 2
assert _test_assignments.loc[[20, 30], "Topic"].isna().all()
assert _test_assignments.loc[20, "Status"] == "Keine Merkmale im Vokabular"
assert _test_assignments.loc[30, "Status"] == "Keine positiven Topic-Gewichte"

status_rows = []
for method, assignments in (("LDA", lda_assignments), ("NMF", nmf_assignments)):
    if not assignments.index.equals(df_model.index):
        raise RuntimeError("Die Reihenfolge der Zuordnungen passt nicht zu den Meldungen.")
    if assignments.loc[~model_mask, "Topic"].notna().any():
        raise RuntimeError("Eine Meldung ohne Merkmale hat unzulässigerweise ein Topic erhalten.")
    assigned_count = int(assignments["Topic"].notna().sum())
    unassigned_count = int(assignments["Topic"].isna().sum())
    assert assigned_count + unassigned_count == N_ANALYSIS_DOCS
    status_rows.append({
        "Verfahren": method,
        "Meldungen nach Text Preprocessing": N_ANALYSIS_DOCS,
        "Topic zugeordnet": assigned_count,
        "Nicht zugeordnet": unassigned_count,
        "Davon ohne Merkmale": int((assignments["Status"] == "Keine Merkmale im Vokabular").sum()),
        "Davon ohne positive Topic-Gewichte": int((assignments["Status"] == "Keine positiven Topic-Gewichte").sum()),
        "Anteil nicht zugeordnet in Prozent": round(100 * unassigned_count / N_ANALYSIS_DOCS, 1),
        "Mehrere Topics mit gleichem Höchstwert": int(assignments["Gleicher Höchstwert"].sum()),
    })
    print(f"{method}: {assigned_count} zugeordnet, {unassigned_count} nicht zugeordnet")

assignment_summary = pd.DataFrame(status_rows)
display(assignment_summary)
print("Kontrolle der Zuordnung: bestanden")


LDA: 2649 zugeordnet, 108 nicht zugeordnet
NMF: 2649 zugeordnet, 108 nicht zugeordnet


,Verfahren,Meldungen nach Text Preprocessing,Topic zugeordnet,Nicht zugeordnet,Davon ohne Merkmale,Davon ohne positive Topic-Gewichte,Anteil nicht zugeordnet in Prozent,Mehrere Topics mit gleichem Höchstwert
0,LDA,2757,2649,108,108,0,3.9,0
1,NMF,2757,2649,108,108,0,3.9,0


Kontrolle der Zuordnung: bestanden


In [30]:
# Schritt 11c Einfluss der Stopwörter vergleichen
# Die Varianten dienen dem Vergleich und ändern die bisherige Analyse nicht.

from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

required_variables = (
    "df_work", "df_model", "df_fit", "df_without_features", "X_count", "X_tfidf",
    "lda_model", "nmf_model", "lda_doc_topic", "nmf_doc_topic",
    "lda_assignments", "nmf_assignments", "count_vectorizer", "tfidf_vectorizer",
    "select_model_documents", "assign_topics", "clean_text",
)
missing_variables = [name for name in required_variables if name not in globals()]
if missing_variables:
    raise RuntimeError(
        "Zuerst die bisherigen Zellen bis einschließlich Schritt 11b ausführen. "
        f"Es fehlen: {', '.join(missing_variables)}"
    )

# Diese im Ausgangszustand entfernten Wörter beschreiben den Zustand oder den Nutzung.
review_status_terms = {
    "defekt", "defekte", "defekten", "defekter",
    "kaputt", "kaputte", "kaputtes", "beschädigt", "beschaedigt",
    "schaden", "schaeden", "schadhaft", "ausgefallen",
    "dunkel", "schwach", "funktion", "funktioniert",
    "brennt", "brennen", "leuchte", "leuchten", "leuchtet",
    "haengt", "hängt", "abgestellt", "abgestellte", "abgestellter",
    "abgestelltes", "abgelegt", "abgeholt", "abholung", "entsorgt",
    "entfernen", "abgelaufen", "abgeschlossen",
}
review_negation_terms = {
    "nicht", "kein", "keine", "keinem", "keinen", "keiner", "keines", "ohne", "nichts",
}
review_original_stopwords = set(STOPWORDS_FINAL)
review_variants = {
    "A Bisherige Stopwörter": review_original_stopwords,
    "B Zustandsangaben behalten": review_original_stopwords - review_status_terms,
    "C Zusätzlich Verneinungen behalten": (
        review_original_stopwords - review_status_terms - review_negation_terms
    ),
}


def clean_text_for_review(text: str, stopwords: set[str]) -> str:
    """Die bisherigen Textregeln mit einer vorgegebenen Menge von Stopwörtern anwenden."""
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-zäöüß\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = []
    for token in text.split():
        token = CANONICAL_TERMS_FINAL.get(token, token)
        if is_location_token(token):
            continue
        if token not in stopwords and len(token) >= MIN_TOKEN_LENGTH:
            tokens.append(token)
    return " ".join(tokens)


# Der Vergleich muss mit denselben Textregeln wie die Hauptanalyse beginnen.
_review_reference = df_work[TEXT_COLUMN].apply(
    lambda text: clean_text_for_review(text, review_original_stopwords)
)
if not _review_reference.equals(df_work[TEXT_COLUMN].apply(clean_text)):
    raise RuntimeError("Die Textregeln des Vergleichs weichen von der Hauptanalyse ab.")

review_rows = []
review_topic_rows = []
review_example_rows = []
review_test_texts = [
    "Lampe defekt", "Ampel funktioniert nicht", "Schild beschädigt",
    "Fahrzeug ohne Kennzeichen", "Fahrrad abgestellt", "Bitte danke",
]

for variant, stopwords in review_variants.items():
    print(f"Prüfe Variante {variant}")
    review_docs = df_work.copy()
    review_docs["text_clean"] = review_docs[TEXT_COLUMN].apply(
        lambda text: clean_text_for_review(text, stopwords)
    )
    review_docs["word_count_clean"] = review_docs["text_clean"].str.split().str.len()
    review_empty = int(review_docs["word_count_clean"].eq(0).sum())
    review_docs = review_docs.loc[
        review_docs["word_count_clean"] >= MIN_TOKENS_CLEAN
    ].reset_index(drop=True)
    if review_docs.empty:
        raise ValueError(f"Variante {variant} enthält keine auswertbaren Texte.")

    cv = CountVectorizer(
        min_df=MIN_DF, max_df=MAX_DF, token_pattern=TOKEN_PATTERN, ngram_range=NGRAM_RANGE
    )
    tv = TfidfVectorizer(
        min_df=MIN_DF, max_df=MAX_DF, token_pattern=TOKEN_PATTERN,
        ngram_range=NGRAM_RANGE, sublinear_tf=SUBLINEAR_TF
    )
    review_count = cv.fit_transform(review_docs["text_clean"])
    review_tfidf = tv.fit_transform(review_docs["text_clean"])
    if not np.array_equal(cv.get_feature_names_out(), tv.get_feature_names_out()):
        raise RuntimeError("Die Vokabulare im Vergleich stimmen nicht überein.")
    review_mask, review_fit, review_count_fit, review_tfidf_fit, review_without = (
        select_model_documents(review_docs, review_count, review_tfidf)
    )
    if N_TOPICS > min(review_tfidf_fit.shape):
        raise ValueError(f"Zu wenige Meldungen oder Merkmale in Variante {variant}.")

    if variant == "A Bisherige Stopwörter":
        if not review_docs[[ID_COLUMN, "text_clean"]].equals(
            df_model[[ID_COLUMN, "text_clean"]]
        ):
            raise RuntimeError("Variante A reproduziert den aktuellen Datenstand nicht.")
        if not np.array_equal(cv.get_feature_names_out(), count_vectorizer.get_feature_names_out()):
            raise RuntimeError("Das Vokabular von Variante A weicht vom aktuellen Stand ab.")
        if (review_count != X_count).nnz or (review_tfidf != X_tfidf).nnz:
            raise RuntimeError("Die Merkmalsmatrizen von Variante A weichen vom aktuellen Stand ab.")
        review_models = {"LDA": lda_model, "NMF": nmf_model}
        review_weights = {"LDA": lda_doc_topic, "NMF": nmf_doc_topic}
    else:
        review_models = {
            "LDA": LatentDirichletAllocation(
                n_components=N_TOPICS, random_state=RANDOM_STATE,
                learning_method="batch", max_iter=LDA_MAX_ITER
            ),
            "NMF": NMF(
                n_components=N_TOPICS, random_state=RANDOM_STATE,
                init="nndsvda", max_iter=NMF_MAX_ITER
            ),
        }
        review_weights = {
            "LDA": review_models["LDA"].fit_transform(review_count_fit),
            "NMF": review_models["NMF"].fit_transform(review_tfidf_fit),
        }

    review_rows.append({
        "Variante": variant,
        "Leere Texte": review_empty,
        "Meldungen nach Text Preprocessing": len(review_docs),
        "Merkmale": review_count.shape[1],
        "Meldungen ohne Merkmale": len(review_without),
        "Meldungen für das Modelltraining": len(review_fit),
    })
    for method, model in review_models.items():
        assignments = assign_topics(review_weights[method], review_mask, review_docs.index)
        if assignments.loc[~review_mask, "Topic"].notna().any():
            raise RuntimeError("Im Vergleich wurde eine Meldung ohne Merkmale einem Topic zugeordnet.")
        feature_names = cv.get_feature_names_out()
        for topic_index, topic_weights in enumerate(model.components_):
            top_indices = topic_weights.argsort()[::-1][:10]
            review_topic_rows.append({
                "Variante": variant,
                "Verfahren": method,
                "Topic": topic_index + 1,
                "Anzahl Meldungen": int(assignments["Topic"].eq(topic_index + 1).sum()),
                "Top-Begriffe": ", ".join(feature_names[top_indices]),
            })
    for text in review_test_texts:
        review_example_rows.append({
            "Variante": variant,
            "Testtext": text,
            "Text nach Preprocessing": clean_text_for_review(text, stopwords),
        })

stopword_review = pd.DataFrame(review_rows)
stopword_review_topics = pd.DataFrame(review_topic_rows)
stopword_review_examples = pd.DataFrame(review_example_rows)

print("\nVergleich der Datengrundlage")
display(stopword_review)
print("\nKünstliche Texte zum Testen, keine Stichprobe aus dem Datensatz")
display(stopword_review_examples)
for variant in review_variants:
    for method in ("LDA", "NMF"):
        print(f"\n{variant} | {method}")
        selected = stopword_review_topics.loc[
            (stopword_review_topics["Variante"] == variant)
            & (stopword_review_topics["Verfahren"] == method),
            ["Topic", "Anzahl Meldungen", "Top-Begriffe"],
        ]
        print(selected.to_string(index=False))

if STOPWORDS_FINAL != review_original_stopwords:
    raise RuntimeError("Die Stopwörter der Hauptanalyse wurden unerwartet verändert.")
print("\nKontrolle des Vergleichs: bestanden")
print("Die Stopwörter und Modelle der Hauptanalyse sind unverändert.")



Prüfe Variante A Bisherige Stopwörter
Prüfe Variante B Zustandsangaben behalten
Prüfe Variante C Zusätzlich Verneinungen behalten

Vergleich der Datengrundlage


,Variante,Leere Texte,Meldungen nach Text Preprocessing,Merkmale,Meldungen ohne Merkmale,Meldungen für das Modelltraining
0,A Bisherige Stopwörter,38,2757,831,108,2649
1,B Zustandsangaben behalten,10,2785,863,73,2712
2,C Zusätzlich Verneinungen behalten,10,2785,869,62,2723



Künstliche Texte zum Testen, keine Stichprobe aus dem Datensatz


,Variante,Testtext,Text nach Preprocessing
0,A Bisherige Stopwörter,Lampe defekt,beleuchtung
1,A Bisherige Stopwörter,Ampel funktioniert nicht,ampel
2,A Bisherige Stopwörter,Schild beschädigt,schild
3,A Bisherige Stopwörter,Fahrzeug ohne Kennzeichen,fahrzeug kennzeichen
4,A Bisherige Stopwörter,Fahrrad abgestellt,fahrrad
5,A Bisherige Stopwörter,Bitte danke,
6,B Zustandsangaben behalten,Lampe defekt,beleuchtung defekt
7,B Zustandsangaben behalten,Ampel funktioniert nicht,ampel funktioniert
8,B Zustandsangaben behalten,Schild beschädigt,schild beschädigt
9,B Zustandsangaben behalten,Fahrzeug ohne Kennzeichen,fahrzeug kennzeichen



A Bisherige Stopwörter | LDA
 Topic  Anzahl Meldungen                                                                                                 Top-Begriffe
     1               258                    müll, mülleimer, straße, illegaler, spielplatz, damen, geehrte, herren, verschmutzt, wald
     2               199           schild, bushaltestelle, fahren, schlagloecher, kreuzung, schule, erkennen, fahrbahn, deckel, laube
     3               257                                  gehweg, fahrrad, rad, laub, fehlt, ende, herrenloses, pfosten, etwa, rechts
     4               228    straße, fahrzeug, kennzeichen, radweg, wollmatingen, oeffentlichen, bahnhof, dettingen, strassenrand, bzw
     5               620           beleuchtung, baum, straße, licht, nachts, betrieb, kreuzung, kinderspielplatz, flackert, repariert
     6               293                                  fahrrad, wasser, verstopft, gulli, läuft, voll, illegal, straße, reifen, ab
     7               251        

## Ergebnis des Vergleichs der Stopwörter

Verglichen wurden drei Varianten bei jeweils zehn Topics. Variante A
verwendet die bisherigen Stopwörter. In Variante B bleiben ausgewählte
Zustandsangaben wie „defekt“ und „abgestellt“ erhalten, in Variante C
zusätzlich Verneinungen wie „nicht“ und „ohne“.

Bei NMF erscheinen die Topics von Variante A anhand ihrer wichtigsten
Begriffe am klarsten nach Sachthemen gegliedert. Die bisherigen Stopwörter
werden deshalb vorerst beibehalten. Die Anzahl der Topics und die
Zuordnung einzelner Meldungen werden anschließend geprüft.ter unverändert.

In [32]:
# Schritt 12 Wichtigste Begriffe je Topic anzeigen

def print_top_words(model, feature_names, n_top_words=12, title=""):
    print("\n" + title)
    print("=" * len(title))

    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[::-1][:n_top_words]
        top_words = [feature_names[i] for i in top_indices]
        print(f"Topic {topic_idx + 1}: {', '.join(top_words)}")

print_top_words(
    lda_model,
    count_vectorizer.get_feature_names_out(),
    n_top_words=12,
    title="LDA Topics auf CountVectorizer"
)

print_top_words(
    nmf_model,
    tfidf_vectorizer.get_feature_names_out(),
    n_top_words=12,
    title="NMF Topics auf TfidfVectorizer"
)



LDA Topics auf CountVectorizer
Topic 1: müll, mülleimer, straße, illegaler, spielplatz, damen, geehrte, herren, verschmutzt, wald, zustand, entlang
Topic 2: schild, bushaltestelle, fahren, schlagloecher, kreuzung, schule, erkennen, fahrbahn, deckel, laube, fahrtrichtung, kommend
Topic 3: gehweg, fahrrad, rad, laub, fehlt, ende, herrenloses, pfosten, etwa, rechts, bushaltestelle, regen
Topic 4: straße, fahrzeug, kennzeichen, radweg, wollmatingen, oeffentlichen, bahnhof, dettingen, strassenrand, bzw, strassenschild, abgemeldetes
Topic 5: beleuchtung, baum, straße, licht, nachts, betrieb, kreuzung, kinderspielplatz, flackert, repariert, gefährlich, entlang
Topic 6: fahrrad, wasser, verstopft, gulli, läuft, voll, illegal, straße, reifen, ab, fahrradstaender, parkplatz
Topic 7: fußgänger, ampel, straße, radfahrer, autos, radweg, grün, gefährlich, fahrradweg, baum, gibt, rot
Topic 8: straße, parkplatz, roller, boden, vermuellt, loch, gueltiges, versicherungsschild, cm, lose, gebuesch, auto


In [33]:
# Schritt 13 Technische Themenübersichten erstellen

def build_topic_summary(model, feature_names, assignments, source_df, n_top_words=12, n_examples=2):
    if not assignments.index.equals(source_df.index):
        raise ValueError("Meldungen und Topic-Zuordnungen haben nicht denselben Index.")
    n_docs = len(source_df)
    if n_docs == 0:
        raise ValueError("Die Bezugsmenge für die Prozentanteile ist leer.")

    rows = []

    for topic_idx in range(model.n_components):
        topic_number = topic_idx + 1

        # Top Begriffe
        topic_weights = model.components_[topic_idx]
        top_indices = topic_weights.argsort()[::-1][:n_top_words]
        top_words = ", ".join(feature_names[top_indices])

        # Dokumente mit dominantem Topic
        doc_indices = np.flatnonzero(
            assignments["Topic"].eq(topic_number).fillna(False).to_numpy(dtype=bool)
        )
        doc_count = len(doc_indices)
        doc_share = round((doc_count / n_docs) * 100, 1)

        # Häufigste bestehende Kategorie zur Plausibilisierung
        if doc_count > 0:
            category_counts = (
                source_df.iloc[doc_indices]["category_parent"]
                .value_counts()
                .head(3)
            )
            top_categories = " | ".join(
                [f"{cat}: {count}" for cat, count in category_counts.items()]
            )

            # Erste zugeordnete Meldungen auswählen und kürzen.
            examples = " || ".join(
                source_df.iloc[doc_indices][TEXT_COLUMN]
                .astype(str)
                .head(n_examples)
                .str.slice(0, 180)
                .tolist()
            )
        else:
            top_categories = ""
            examples = ""

        rows.append({
            "Topic": topic_number,
            "Anzahl Meldungen": doc_count,
            "Anteil Prozent": doc_share,
            "Bezugsmenge Meldungen": n_docs,
            "Top Begriffe": top_words,
            "Häufigste Kategorie": top_categories,
            "Beispielmeldungen": examples
        })

    summary = pd.DataFrame(rows)
    summary = summary.sort_values("Anzahl Meldungen", ascending=False).reset_index(drop=True)
    summary.insert(0, "Rang", np.arange(1, len(summary) + 1))

    if int(summary["Anzahl Meldungen"].sum()) != int(assignments["Topic"].notna().sum()):
        raise RuntimeError("Die Themenhäufigkeiten stimmen nicht mit den Zuordnungen überein.")
    return summary

lda_summary = build_topic_summary(
    lda_model,
    count_vectorizer.get_feature_names_out(),
    lda_assignments,
    df_model
)

nmf_summary = build_topic_summary(
    nmf_model,
    tfidf_vectorizer.get_feature_names_out(),
    nmf_assignments,
    df_model
)

print("LDA Themenübersicht:")
display(lda_summary)

print("NMF Themenübersicht:")
display(nmf_summary)

print("Die Anteile beziehen sich auf alle", N_ANALYSIS_DOCS, "Meldungen nach Text Preprocessing.")
print("Nicht zugeordnete Meldungen sind in der separaten Zuordnungsübersicht enthalten.")
print("Gerundete Prozentanteile können sich geringfügig von 100 Prozent unterscheiden.")


LDA Themenübersicht:


,Rang,Topic,Anzahl Meldungen,Anteil Prozent,Bezugsmenge Meldungen,Top Begriffe,Häufigste Kategorie,Beispielmeldungen
0,1,5,620,22.5,2757,"beleuchtung, baum, straße, licht, nachts, betr...","Beleuchtung: 555 | Strassen, Fahrrad & Verkehr...",Strassenlaterne zwischen Nr. 17A und 19 seit T...
1,2,6,293,10.6,2757,"fahrrad, wasser, verstopft, gulli, läuft, voll...","Strassen, Fahrrad & Verkehr: 187 | Muell/Saube...",Seit der Nacht von Sonntag auf Montag steht im...
2,3,1,258,9.4,2757,"müll, mülleimer, straße, illegaler, spielplatz...","Muell/Sauberkeit: 171 | Strassen, Fahrrad & Ve...","Glas und Muell || Guten Tag, bevor das Csmpus ..."
3,4,3,257,9.3,2757,"gehweg, fahrrad, rad, laub, fehlt, ende, herre...","Strassen, Fahrrad & Verkehr: 132 | Muell/Saube...",Eine Parkbank wurde in der Nacht von Freitag a...
4,5,7,251,9.1,2757,"fußgänger, ampel, straße, radfahrer, autos, ra...","Strassen, Fahrrad & Verkehr: 132 | Pflanzenwuc...",permanentes rausstellen von Grobmuell/ Altklei...
5,6,4,228,8.3,2757,"straße, fahrzeug, kennzeichen, radweg, wollmat...","Strassen, Fahrrad & Verkehr: 135 | Muell/Saube...",Abgemeldetes Moped am Strassenrand (die kleine...
6,7,8,217,7.9,2757,"straße, parkplatz, roller, boden, vermuellt, l...","Strassen, Fahrrad & Verkehr: 95 | Muell/Sauber...",Zwischen Hardergasse 6 bis 10 sind mehrere Ran...
7,8,2,199,7.2,2757,"schild, bushaltestelle, fahren, schlagloecher,...","Strassen, Fahrrad & Verkehr: 108 | Muell/Saube...",Schild umgestuerzt. || An der bushaltestelle T...
8,9,10,165,6.0,2757,"radweg, sperrmuell, scherben, schild, mittlerw...","Strassen, Fahrrad & Verkehr: 60 | Muell/Sauber...",In der Skaterbahn des urisberg Spielplatz befi...
9,10,9,161,5.8,2757,"glasscherben, platz, entfernt, hoch, besteht, ...","Muell/Sauberkeit: 63 | Strassen, Fahrrad & Ver...",Hecke ragt in Fussgaengerweg. Das angebrachte ...


NMF Themenübersicht:


,Rang,Topic,Anzahl Meldungen,Anteil Prozent,Bezugsmenge Meldungen,Top Begriffe,Häufigste Kategorie,Beispielmeldungen
0,1,1,572,20.7,2757,"beleuchtung, kurve, licht, weitere, flackert, ...","Beleuchtung: 561 | Strassen, Fahrrad & Verkehr...",Strassenlaterne zwischen Nr. 17A und 19 seit T...
1,2,8,488,17.7,2757,"ampel, fußgänger, grün, rot, radfahrer, kreuzu...","Strassen, Fahrrad & Verkehr: 258 | Muell/Saube...",Seit der Nacht von Sonntag auf Montag steht im...
2,3,4,229,8.3,2757,"straße, entlang, abgemeldeter, wasser, scherbe...","Strassen, Fahrrad & Verkehr: 100 | Muell/Saube...",permanentes rausstellen von Grobmuell/ Altklei...
3,4,7,228,8.3,2757,"glasscherben, radweg, fahrradweg, scherben, bu...","Muell/Sauberkeit: 102 | Strassen, Fahrrad & Ve...",In der Skaterbahn des urisberg Spielplatz befi...
4,5,3,221,8.0,2757,"fahrrad, herrenloses, sattel, bewegt, platz, d...","Strassen, Fahrrad & Verkehr: 185 | Muell/Saube...",Fahrrad im Wald bei DorfweiherAngekettet an un...
5,6,2,215,7.8,2757,"müll, illegaler, mülleimer, illegal, wald, spi...","Muell/Sauberkeit: 203 | Strassen, Fahrrad & Ve...","Glas und Muell || Guten Tag, bevor das Csmpus ..."
6,7,5,207,7.5,2757,"gehweg, wächst, hecke, zugewachsen, rad, ragt,...","Muell/Sauberkeit: 56 | Strassen, Fahrrad & Ver...",Nicht abgeholte/ ausgetragene Konstanzer Anzei...
7,8,9,199,7.2,2757,"baum, quer, umgestürzt, rad, matratze, äste, s...","Strassen, Fahrrad & Verkehr: 66 | Muell/Sauber...",Brombeeren wachsen quer ueber den Weg. Verletz...
8,9,6,177,6.4,2757,"schild, umgefahren, boden, ausgeblichen, angef...","Strassen, Fahrrad & Verkehr: 116 | Muell/Saube...","Schild umgestuerzt. || Verkehrsschild ""Ueberho..."
9,10,10,113,4.1,2757,"roller, versicherungsschild, gueltiges, kennze...","Strassen, Fahrrad & Verkehr: 93 | Muell/Sauber...",Vor dem Haus Eberplatz 6 steht ein Roller ohne...


Die Anteile beziehen sich auf alle 2757 Meldungen nach Text Preprocessing.
Nicht zugeordnete Meldungen sind in der separaten Zuordnungsübersicht enthalten.
Gerundete Prozentanteile können sich geringfügig von 100 Prozent unterscheiden.


In [34]:
# Schritt 14 Technische Themenübersichten als CSV speichern



# Die Spalte mit Rohtextbeispielen nicht in diesen CSV-Export übernehmen.
# Beispielmeldungen können Adressen oder personenbezogene Hinweise enthalten.
lda_export = lda_summary.drop(columns=["Beispielmeldungen"], errors="ignore")
nmf_export = nmf_summary.drop(columns=["Beispielmeldungen"], errors="ignore")

lda_path = OUTPUT_DIR / "themenuebersicht_lda.csv"
nmf_path = OUTPUT_DIR / "themenuebersicht_nmf.csv"

lda_export.to_csv(lda_path, index=False, encoding="utf-8-sig")
nmf_export.to_csv(nmf_path, index=False, encoding="utf-8-sig")

print("Gespeichert:")
print(lda_path.resolve())
print(nmf_path.resolve())

assignment_path = OUTPUT_DIR / "zuordnung_lda_nmf.csv"
assignment_summary.to_csv(assignment_path, index=False, sep=";", encoding="utf-8-sig")
print(assignment_path.resolve())


Gespeichert:
C:\Users\Seb\Documents\GitHub\portfolio-data-analysis\outputs\themenuebersicht_lda.csv
C:\Users\Seb\Documents\GitHub\portfolio-data-analysis\outputs\themenuebersicht_nmf.csv
C:\Users\Seb\Documents\GitHub\portfolio-data-analysis\outputs\zuordnung_lda_nmf.csv


In [35]:
# Schritt 15 NMF-Ergebnisse mit Themenbeschreibungen ergänzen

# Die bisherigen Bezeichnungen sind für das geänderte Modell noch nicht geprüft.
decision_table = None
if TOPIC_LABELS_REVIEWED:

    from sklearn.feature_extraction.text import CountVectorizer
    from pathlib import Path
    import numpy as np
    import pandas as pd

    # Bereits berechnete NMF-Zuordnungen übernehmen
    df_topic_docs = df_model.copy()
    df_topic_docs["NMF_Topic"] = nmf_assignments["Topic"]

    # Manuell formulierte Themenbezeichnungen, Beschreibungen und beispielhafte Hinweise.
    # Die Zuordnung gilt nur für den bisherigen Modellstand mit zehn Topics.
    # Nach erneutem Training ist die inhaltliche Zuordnung der Labels zu prüfen.
    # Die beispielhaften Hinweise sind keine wörtlichen Originalmeldungen.
    topic_meta = {
        1: {
            "Beschwerdethema": "Defekte Beleuchtung",
            "Kurzbeschreibung": "Viele Meldungen betreffen ausgefallene, flackernde oder schlecht sichtbare Beleuchtung im öffentlichen Raum.",
            "Typische Hinweise": "Laterne ausgefallen, Licht flackert, dunkler Bereich"
        },
        2: {
            "Beschwerdethema": "Herrenlose Fahrräder",
            "Kurzbeschreibung": "Viele Meldungen betreffen Fahrräder, die längere Zeit unbewegt stehen, beschädigt sind oder öffentliche Flächen blockieren.",
            "Typische Hinweise": "Fahrrad steht seit Wochen, Sattel fehlt, Fahrrad blockiert Platz"
        },
        3: {
            "Beschwerdethema": "Müll und Sauberkeit",
            "Kurzbeschreibung": "Viele Hinweise betreffen illegal abgelegten Müll, überfüllte Mülleimer und Verschmutzungen im öffentlichen Raum.",
            "Typische Hinweise": "Müll abgeladen, Mülleimer voll, Abfall im Gebüsch"
        },
        4: {
            "Beschwerdethema": "Gehwege und Heckenwuchs",
            "Kurzbeschreibung": "Meldungen betreffen zugewachsene Gehwege, Hecken, Engstellen und Einschränkungen für Fußgänger.",
            "Typische Hinweise": "Hecke wächst in Gehweg, Gehweg zu eng, Weg zugewachsen"
        },
        5: {
            "Beschwerdethema": "Straßenraum, Fahrzeuge und Behinderungen",
            "Kurzbeschreibung": "Das Thema bündelt Meldungen zu Straßen, Einfahrten, Parkplätzen, abgestellten Fahrzeugen und Behinderungen im Verkehrsraum.",
            "Typische Hinweise": "Fahrzeug abgestellt, Straße blockiert, Einfahrt beeinträchtigt"
        },
        6: {
            "Beschwerdethema": "Beschilderung",
            "Kurzbeschreibung": "Dieses Thema umfasst beschädigte, verschmutzte, umgefahrene oder schlecht erkennbare Schilder.",
            "Typische Hinweise": "Schild umgestürzt, Verkehrsschild verschmutzt, Schild schlecht sichtbar"
        },
        7: {
            "Beschwerdethema": "Radwege, Entwässerung und Gefahrenstellen",
            "Kurzbeschreibung": "Mehrere Meldungen betreffen Radwege und Wege mit Wasserstellen, verstopften Abläufen, Scherben oder anderen Gefahrenstellen.",
            "Typische Hinweise": "Wasser auf Radweg, Gulli verstopft, Scherben auf Weg"
        },
        8: {
            "Beschwerdethema": "Bäume, Äste und sonstige Hindernisse",
            "Kurzbeschreibung": "Meldungen beschreiben umgestürzte Bäume, überhängende Äste, Bewuchs oder Gegenstände, die Wege und Flächen blockieren.",
            "Typische Hinweise": "Baum umgestürzt, Äste hängen tief, Weg blockiert"
        },
        9: {
            "Beschwerdethema": "Ampeln und Fußgängerquerungen",
            "Kurzbeschreibung": "Meldungen beziehen sich auf Ampeln, Rot- und Grünphasen sowie Konflikte zwischen Fußgängern, Radfahrern und Verkehr.",
            "Typische Hinweise": "Ampel defekt, Rotphase zu lang, Querung unsicher"
        },
        10: {
            "Beschwerdethema": "Glasscherben und Verschmutzung an öffentlichen Orten",
            "Kurzbeschreibung": "Viele Meldungen betreffen Scherben und Verschmutzungen auf Spielplätzen, Radwegen, an Bushaltestellen oder auf öffentlichen Flächen.",
            "Typische Hinweise": "Glasscherben auf Spielplatz, Scherben auf Radweg, Müll an Haltestelle"
        }
    }

    # Prüfen, ob für jede Topic-Nummer eine Themenbezeichnung hinterlegt ist.
    # Diese Prüfung bewertet nicht, ob die Bezeichnungen inhaltlich passend sind.
    model_topics = set(nmf_summary["Topic"].astype(int))
    label_topics = set(topic_meta.keys())

    missing_labels = sorted(model_topics - label_topics)
    unused_labels = sorted(label_topics - model_topics)

    if missing_labels:
        raise ValueError(f"Für diese Topics fehlen Labels: {missing_labels}")

    if unused_labels:
        print(f"Hinweis: Diese Labels werden aktuell nicht genutzt: {unused_labels}")


    def top_child_categories(series, n=3):
        counts = series.value_counts().head(n)
        return "; ".join([f"{category} ({count})" for category, count in counts.items()])


    # Kombinationen aus bereinigten Tokens sind nicht zwangsläufig Originalphrasen.
    def top_ngrams_for_topic(texts, top_n=5):
        texts = [text for text in texts if isinstance(text, str) and len(text.split()) >= 2]

        if len(texts) == 0:
            return ""

        min_df_value = 2 if len(texts) >= 5 else 1

        vectorizer = CountVectorizer(
            ngram_range=(2, 3),
            min_df=min_df_value,
            token_pattern=r"(?u)\b[a-zäöüß][a-zäöüß]+\b"
        )

        try:
            X = vectorizer.fit_transform(texts)
        except ValueError:
            return ""

        terms = vectorizer.get_feature_names_out()
        scores = np.asarray(X.sum(axis=0)).ravel()

        top_indices = scores.argsort()[::-1][:top_n]
        return "; ".join([terms[i] for i in top_indices])


    # Dominante Unterkategorien je Topic berechnen
    child_summary = (
        df_topic_docs
        .groupby("NMF_Topic")["category_child"]
        .apply(top_child_categories)
        .reset_index()
        .rename(columns={
            "NMF_Topic": "Topic",
            "category_child": "Dominante Unterkategorien"
        })
    )

    # Häufige Kombinationen aus zwei oder drei Tokens im bereinigten Text zählen.
    phrase_rows = []

    for topic_id in sorted(model_topics):
        topic_texts = df_topic_docs.loc[
            df_topic_docs["NMF_Topic"] == topic_id,
            "text_clean"
        ].tolist()

        phrase_rows.append({
            "Topic": topic_id,
            "Auffällige Wortkombinationen": top_ngrams_for_topic(topic_texts, top_n=5)
        })

    phrase_summary = pd.DataFrame(phrase_rows)

    # Manuell hinterlegte Themenangaben in einen DataFrame überführen.
    topic_meta_df = (
        pd.DataFrame.from_dict(topic_meta, orient="index")
        .rename_axis("Topic")
        .reset_index()
    )

    # Entscheidungsträger-Tabelle bauen
    decision_table = (
        nmf_summary[["Topic", "Anzahl Meldungen", "Anteil Prozent"]]
        .merge(topic_meta_df, on="Topic", how="left")
        .merge(child_summary, on="Topic", how="left")
        .merge(phrase_summary, on="Topic", how="left")
    )

    decision_table = decision_table.sort_values(
        "Anzahl Meldungen",
        ascending=False
    ).reset_index(drop=True)

    decision_table.insert(0, "Rang", np.arange(1, len(decision_table) + 1))

    decision_table["Anteil"] = decision_table["Anteil Prozent"].map(
        lambda value: f"{value:.1f} %"
    )

    decision_table = decision_table[
        [
            "Rang",
            "Beschwerdethema",
            "Kurzbeschreibung",
            "Anzahl Meldungen",
            "Anteil",
            "Dominante Unterkategorien",
            "Typische Hinweise",
            "Auffällige Wortkombinationen"
        ]
    ]

    display(decision_table)
else:
    print("Beschriftete Ergebnisübersicht und Diagramm bleiben bis zur Prüfung der Themenbezeichnungen deaktiviert.")


Beschriftete Ergebnisübersicht und Diagramm bleiben bis zur Prüfung der Themenbezeichnungen deaktiviert.


In [36]:
# Schritt 16 Übersicht für Entscheidungstragende aufbereiten

decision_table_final = None
if decision_table is not None:

    decision_table_final = decision_table.copy()

    # Spalten für die Übersicht für Entscheidungstragende auswählen.
    # Die zur Plausibilisierung verwendeten Kategorien bleiben in der technischen Übersicht.
    decision_table_final = decision_table_final[
        [
            "Rang",
            "Beschwerdethema",
            "Kurzbeschreibung",
            "Anzahl Meldungen",
            "Anteil",
            "Typische Hinweise"
        ]
    ].copy()

    # Ausgewählte Themenbezeichnungen an die bisherige inhaltliche Interpretation anpassen.
    decision_table_final["Beschwerdethema"] = decision_table_final["Beschwerdethema"].replace({
        "Radwege, Entwässerung und Gefahrenstellen": "Gefahrenstellen auf Wegen und Radwegen",
        "Bäume, Äste und sonstige Hindernisse": "Hindernisse im öffentlichen Raum",
        "Glasscherben und Verschmutzung an öffentlichen Orten": "Scherben und Verschmutzung an öffentlichen Orten"
    })

    # Kurzbeschreibungen an die finalen Labels anpassen
    decision_table_final.loc[
        decision_table_final["Beschwerdethema"] == "Gefahrenstellen auf Wegen und Radwegen",
        "Kurzbeschreibung"
    ] = "Mehrere Meldungen betreffen Gefahrenstellen auf Wegen und Radwegen, darunter Scherben, Wasser, verstopfte Abläufe oder Hindernisse."

    decision_table_final.loc[
        decision_table_final["Beschwerdethema"] == "Hindernisse im öffentlichen Raum",
        "Kurzbeschreibung"
    ] = "Meldungen beschreiben blockierte oder eingeschränkte öffentliche Flächen, zum Beispiel durch Bäume, Äste, abgestellte Gegenstände oder andere Hindernisse."

    decision_table_final.loc[
        decision_table_final["Beschwerdethema"] == "Scherben und Verschmutzung an öffentlichen Orten",
        "Kurzbeschreibung"
    ] = "Viele Meldungen betreffen Scherben und Verschmutzungen auf Spielplätzen, Radwegen, an Bushaltestellen oder auf öffentlichen Flächen."

    # Textfelder bereinigen
    decision_table_final["Typische Hinweise"] = decision_table_final["Typische Hinweise"].astype(str).str.strip()
    decision_table_final["Anteil"] = decision_table_final["Anteil"].astype(str).str.strip()

    # Spaltenname präzisieren
    decision_table_final = decision_table_final.rename(
        columns={"Typische Hinweise": "Beispielhafte Hinweise"}
    )

    display(decision_table_final)

if decision_table_final is not None:
    print("Bezugsmenge:", N_ANALYSIS_DOCS, "Meldungen nach Text Preprocessing.")
    print("Nicht zugeordnet:", int(nmf_assignments["Topic"].isna().sum()))


In [37]:
# Schritt 17 Übersicht für Entscheidungstragende speichern

if decision_table_final is not None:

    decision_table_final_path = OUTPUT_DIR / "entscheidungstraeger_themenuebersicht_final.csv"

    decision_table_final.to_csv(
        decision_table_final_path,
        index=False,
        sep=";",
        encoding="utf-8-sig"
    )

    print("Finale Entscheidungsträger-Übersicht gespeichert:")
    print(decision_table_final_path.resolve())


In [38]:
# Schritt 18 Themenhäufigkeiten als Balkendiagramm darstellen

if decision_table_final is not None:

    import matplotlib.pyplot as plt

    chart_df = decision_table_final.sort_values("Anzahl Meldungen", ascending=True)

    plt.figure(figsize=(10, 6))
    plt.barh(chart_df["Beschwerdethema"], chart_df["Anzahl Meldungen"])
    plt.xlabel("Anzahl Meldungen")
    plt.ylabel("Beschwerdethema")
    plt.title("Häufigste Beschwerdethemen im Mängelmelder Konstanz")
    plt.tight_layout()

    chart_path = OUTPUT_DIR / "haeufigste_beschwerdethemen_final.png"
    plt.savefig(chart_path, dpi=150)
    plt.show()

    print("Diagramm gespeichert:")
    print(chart_path.resolve())
